**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to RAPIDS

> ⚠️ **Draft — requires an NVIDIA GPU; code cells not executed here.** This notebook was authored on a machine without CUDA, so unlike most workshops its cells ship without saved outputs. Run it on a CUDA machine (or [Colab](https://colab.research.google.com/) with a GPU runtime, where RAPIDS installs in ~2 min) and an instructor should verify each cell before teaching. Remove this banner after that pass.

The [GPU workshop](./Intro_GPU.ipynb) accelerated *arrays* with CuPy. RAPIDS extends the same move up the stack: **cuDF** is pandas on the GPU, **cuML** is scikit-learn on the GPU — same APIs, different silicon. The workshop's real subject is *benchmarking honestly*: when does the GPU actually win, and when are you just paying the PCIe toll?

## 0. Introduction

RAPIDS exists because dataframe work — parse, filter, join, group — is data-parallel too. The same caveat from [Intro to GPU Systems §3.3](./Intro_GPU.ipynb) governs everything: **transfers dominate**. The wins come from keeping the whole pipeline on-device.

## 1. Pre-requisites

- [Intro to Python](../Intro_Programming/Intro_Python/Intro_Python.ipynb) (NumPy); pandas basics helpful.
- [Intro to GPU Systems](./Intro_GPU.ipynb) — the host/device mental model is assumed.
- Install (conda, CUDA 12.x): `conda create -n rapids -c rapidsai -c conda-forge -c nvidia rapids=24.06 python=3.11 cuda-version=12.2`
- On Colab: GPU runtime, then follow the [RAPIDS Colab install](https://rapids.ai/) cell.

---
### 🕐 Session 1 of 2 — *cuDF: Dataframes on the Device* (~35 min)
**Goal:** port a pandas pipeline to cuDF; benchmark honestly, transfers included.
**Builds on:** [Intro to GPU Systems](./Intro_GPU.ipynb). &nbsp; **Feeds into:** Session 2 (cuML).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: cuDF — Dataframes on the Device</b></summary>

**Timing (~35 min).** 5 min what RAPIDS is · 10 min the port · 15 min the honest benchmark · 5 min the moral.

**First, the practical warning: this notebook ships without outputs and needs a GPU.** The banner says so. **Run every cell yourself on a CUDA machine before teaching** — a Colab GPU runtime installs RAPIDS in about two minutes and is the lowest-friction option. Teaching from unexecuted cells means discovering an API change live, and RAPIDS moves fast.

**Frame RAPIDS in one sentence and then move on, because the API is deliberately unsurprising.** cuDF is pandas on the GPU; cuML is scikit-learn on the GPU. **The APIs are copied on purpose** so the port costs almost nothing. That is the selling point and it is also the trap: *porting* is cheap, and whether the *run* is faster is an entirely separate question.

**Which is why the real subject of this workshop is benchmarking, not RAPIDS.** Say that explicitly. Anyone can rewrite `pd` as `cudf`; the skill being taught is **knowing whether it helped**, and the notebook is built around a comparison that flatters the GPU followed by one that does not.

**Make the first benchmark's flaw the teaching moment.** In §2, `gdf` was **already resident on the device** — the transfer happened in the previous cell and is not in the measured region. **That is the most common way GPU benchmarks lie**, and it is rarely deliberate: you convert once at the top of a notebook and then time the pipeline. Ask the room where the transfer went before pointing at `gdf = cudf.from_pandas(pdf)`.

**Praise the `np.allclose` check while you are there.** The benchmark verifies that the two pipelines produce the **same answer** before comparing their speeds. **Fast and wrong is the default failure of accelerated code**, and a timing without a correctness check is not a result. This is the same discipline the [Intro_GPU](./Intro_GPU.ipynb) Numba example follows and its CuPy cells do not.

**Then §2.1 is the honest version, and the sweep is the point.** Charging `cudf.from_pandas` inside the timed region at $n = 10^4$, $10^6$, $10^7$ should show **pandas winning at small $n$ and cuDF winning at large $n$**, with a crossover in between. **Have the room predict the crossover before running.** Guesses are usually far too low, because people underestimate the fixed PCIe cost.

**Give them the arithmetic to predict with, so the answer is principled rather than merely observed.** At ~5 GB/s, transferring $n$ rows of three columns (about 20 bytes each) costs roughly $4n$ nanoseconds; a pandas groupby over $n$ rows costs on the order of $50n$ nanoseconds. **The crossover is wherever the compute saving exceeds the transfer** — order $10^5$–$10^6$ rows on typical hardware, and it moves with how much work the pipeline does per row.

**Which leads to the moral, and it is the one design rule worth remembering.** The transfer is paid **once**; the compute saving accrues **per operation**. So the fix is not a faster bus — it is **a longer pipeline on the device**. Load with `cudf.read_parquet` straight onto the GPU, do every filter, join, and aggregation there, and bring back only the summary. **Round-tripping through pandas between steps destroys the entire advantage**, and it is by far the most common mistake in real RAPIDS code.

**One thing to flag if the room is sharp.** `cudf.from_pandas` on 10 million rows also *allocates* 200+ MB of device memory, and GPU memory is far scarcer than host RAM. **Out-of-memory is a more common RAPIDS failure than disappointing speed**, and `dask-cudf` exists to chunk work that does not fit — worth naming even though this workshop does not use it.
</details>

## 2. cuDF

💡 **Intuition.** cuDF stores each column as a contiguous GPU array (Apache Arrow layout) — a groupby becomes thousands of threads binning rows in parallel. The API is deliberately pandas-shaped so the *port* is cheap; whether the *run* is faster depends entirely on data size and transfer count, which is why we benchmark before believing.

In [ ]:
import numpy as np
import pandas as pd
import cudf                       # GPU dataframe — pandas-shaped on purpose
import time

# Synthetic sensor log, in the spirit of the Databases workshop schema
N = 10_000_000
rng = np.random.default_rng(0)
pdf = pd.DataFrame({
    "sensor_id": rng.integers(0, 200, N),
    "value":     rng.standard_normal(N),
    "quality":   rng.integers(0, 4, N),
})
gdf = cudf.from_pandas(pdf)        # ← host→device transfer: this line COSTS
print(type(gdf), len(gdf))

In [ ]:
def pipeline(df):
    good = df[df.quality >= 2]
    stats = good.groupby("sensor_id").value.agg(["mean", "std", "count"])
    return stats.sort_index()

tic = time.perf_counter(); cpu_out = pipeline(pdf);  t_cpu = time.perf_counter() - tic
tic = time.perf_counter(); gpu_out = pipeline(gdf);  t_gpu = time.perf_counter() - tic
# fair check: results must MATCH (bring GPU result back for comparison)
ok = np.allclose(cpu_out["mean"].values, gpu_out["mean"].to_pandas().values, atol=1e-6)
print(f"pandas: {t_cpu:.3f} s   cuDF: {t_gpu:.3f} s   results match: {ok}")

### 2.1. The Honest Benchmark

The timing above *flatters* the GPU: `gdf` was already resident. An end-to-end comparison must charge the transfer — and small data flips the verdict:

In [ ]:
for n in [10_000, 1_000_000, 10_000_000]:
    small = pdf.iloc[:n]
    tic = time.perf_counter()
    _ = pipeline(cudf.from_pandas(small))      # transfer + compute, all charged
    t_gpu_e2e = time.perf_counter() - tic
    tic = time.perf_counter()
    _ = pipeline(small)
    t_cpu2 = time.perf_counter() - tic
    print(f"n = {n:>10,}:  pandas {t_cpu2:.4f} s   cuDF end-to-end {t_gpu_e2e:.4f} s")
# Expect: pandas wins small n; cuDF wins large n. Find YOUR crossover.

**What just happened.** The same pipeline, timed three times at three data sizes — with the **transfer charged inside the measured region** this time. Expect pandas to win at $n = 10^4$, cuDF to win at $n = 10^7$, and a crossover somewhere between.

> ⚠️ This notebook ships **without saved outputs** (see the banner), so the numbers are yours to produce. Run it and fill in your own crossover before drawing conclusions.

**The one-line difference from §2 is the whole point.** There, `gdf` was already on the device — the transfer happened in a previous cell and was invisible to the clock. Here `cudf.from_pandas(small)` sits **inside** the timed block. **Same pipeline, same hardware, different verdict**, and the only change is where the stopwatch starts.

**That is the most common way GPU benchmarks mislead, and it is usually accidental.** You convert your data once at the top of a notebook, then time the interesting part. Nothing is dishonest about any individual line; the *comparison* is simply not measuring what its label claims. **Ask of any accelerator benchmark: where did the data come from, and was getting it there charged?**

**Predict the crossover from arithmetic before reading it off the output, because that makes it a principle rather than a datum.** At roughly 5 GB/s over PCIe, moving $n$ rows of three columns (about 20 bytes each) costs on the order of $4n$ nanoseconds. A pandas groupby over $n$ rows costs on the order of $50n$ nanoseconds. **The GPU wins once its compute saving exceeds that fixed toll** — typically $10^5$–$10^6$ rows, and the threshold moves with how much work the pipeline does per row.

**Which yields the design rule this session exists to deliver.** The transfer is paid **once**; the compute saving accrues **per operation**. So the answer is never a faster bus — it is **a longer pipeline on the device**. Load with `cudf.read_parquet` directly onto the GPU, do every filter, join, and aggregation there, and bring back only the summary. **Round-tripping through pandas between steps forfeits the entire advantage**, and doing so accidentally is the most common mistake in real RAPIDS code.

**Two details in the loop worth noticing before you trust your own numbers.** There is no **warm-up**: the first CUDA call in a process pays context initialisation, so the $n = 10{,}000$ row may be measuring the driver rather than the work — exactly the effect the [Intro_GPU](./Intro_GPU.ipynb) transfer cells ran into. And these are **single runs**; a fair sweep repeats each size and takes the minimum, which is what the [Performance Engineering](./Performance_Engineering.ipynb) `bench` helper does.

**Finally, the constraint that decides more RAPIDS projects than speed does.** `cudf.from_pandas` at $n = 10^7$ also **allocates** 200+ MB of device memory, and GPU memory is far scarcer than host RAM. **Out-of-memory is a more common failure than disappointing performance**, and `dask-cudf` exists to chunk work that does not fit on the card.

Moral (same as [GPU workshop §3.3](./Intro_GPU.ipynb)): the GPU wins when data is big and **stays on the device across the whole pipeline** — load with `cudf.read_parquet` directly on the GPU rather than round-tripping through pandas.

---
### 🕐 Session 2 of 2 — *cuML: Machine Learning on the Device* (~35 min)
**Goal:** GPU-accelerate scikit-learn-style fits; know the decision rule for when RAPIDS earns its keep.
**Builds on:** Session 1.

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: cuML — Machine Learning on the Device</b></summary>

**Timing (~35 min).** 10 min why K-means suits a GPU · 12 min the benchmark · 8 min the decision rule · 5 min limits.

**Open by asking *why* K-means should accelerate well, rather than asserting that it does.** Each iteration computes the distance from every point to every centroid — $2{,}000{,}000 \times 8$ dense, regular, independent operations, repeated until convergence. **No branching, no irregular memory access, the same arithmetic on millions of elements.** That is exactly the shape a GPU is built for, and deriving it beats being told.

**Then generalise the property, because that is the transferable part.** cuML wins on **iterative, matrix-heavy** fits — K-means, PCA, linear and logistic regression, UMAP, random forests. It adds little when the model is small, the fit is I/O-bound, or the algorithm is inherently sequential. **The question is never "is this in cuML?" but "is this compute-bound and large?"**

**Point at `X_dev = cp.asarray(X_host)` and say why the notebook writes it as its own line.** Session 1's first benchmark hid its transfer; this one makes it visible and separate. **Naming the cost is how you stop forgetting to charge it.** The fit is then timed *after* the transfer — which is the right choice if the question is "how fast is the fit" and the wrong one if the question is "should I use a GPU". **Being explicit about which question you are answering is the whole discipline.**

**Flag the fairness issue in the comparison, since it is real and easy to miss.** scikit-learn's K-means is heavily optimised and multithreaded, `n_init=1` disables its usual restarts, and the two libraries need not share an initialisation scheme or convergence tolerance. **A speed comparison between two implementations of an approximate algorithm is meaningless without a quality check** — which is why the cell prints the relative inertia agreement. **Read that line before the timing line.**

**Then give the decision rule as the workshop's deliverable, and make it a checklist rather than a slogan.** RAPIDS earns its keep when **(1)** the data is millions of rows, **(2)** the pipeline stays on-device end to end, and **(3)** the algorithm is compute-bound. **Miss any one and pandas/sklearn are simpler and often faster.** Ask the room to apply it to three problems they actually work on; the answer is "no" more often than they expect, and that is the useful outcome.

**Reinforce with the constraint that binds most often in practice.** Two million rows by eight float32 columns is 64 MB — comfortable — but a 24 GB card fills quickly once intermediates accumulate. **Out-of-memory is a more common RAPIDS failure than disappointing speed**, and `dask-cudf` exists to chunk work that does not fit.

**Close by connecting the two sessions and pointing forward.** Session 1: transfers dominate, so keep the pipeline resident. Session 2: even resident, the algorithm must be compute-bound to benefit. **Together they are the [Intro_GPU](./Intro_GPU.ipynb) CGMA argument one level up the stack** — and [Performance Engineering](./Performance_Engineering.ipynb) turns "is this compute-bound?" from a judgement call into a measurement on a roofline chart.
</details>

## 3. cuML

In [ ]:
from cuml.cluster import KMeans as cuKMeans
from sklearn.cluster import KMeans as skKMeans
import cupy as cp

X_host = rng.standard_normal((2_000_000, 8)).astype(np.float32)
X_dev  = cp.asarray(X_host)                      # explicit, charged transfer

tic = time.perf_counter()
sk = skKMeans(n_clusters=8, n_init=1, random_state=0).fit(X_host)
t_sk = time.perf_counter() - tic

tic = time.perf_counter()
cu = cuKMeans(n_clusters=8, n_init=1, random_state=0).fit(X_dev)
t_cu = time.perf_counter() - tic

print(f"scikit-learn: {t_sk:.2f} s   cuML: {t_cu:.2f} s")
print(f"inertia agreement (relative): {abs(sk.inertia_ - float(cu.inertia_)) / sk.inertia_:.2e}")

💡 **Intuition.** K-means is distance computations in a loop — exactly the dense, regular arithmetic GPUs devour. The pattern generalizes: cuML shines on **iterative, matrix-heavy** fits (KMeans, PCA, linear/logistic, UMAP, random forests) and adds little when the model is tiny or the fit is I/O-bound.

**Decision rule to teach:** (1) data ≥ millions of rows, (2) pipeline stays on-device end to end, (3) the algorithm is compute-bound → RAPIDS. Otherwise pandas/sklearn are simpler and often faster. Measure, don't assume — you now know how to measure honestly.

## 4. Conclusion

cuDF and cuML move the [GPU workshop's](./Intro_GPU.ipynb) lesson up the stack: same APIs you know, massive wins **iff** data is large and transfers are amortized. The benchmark discipline — charge the transfer, verify the outputs match, find the crossover — is the transferable skill.

---
## Where next

- [Intro to Databases](../Intro_Host_Prog/Intro_Databases/Intro_Databases.ipynb) — "arrays in files, metadata in SQL" pairs with `cudf.read_parquet` beautifully.
- [Scaling Neural Networks](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — the same bottleneck-hunting mindset applied to training.
- [Intro to GPU Systems](./Intro_GPU.ipynb) — the memory model underneath every timing above.